In [ ]:
import traceback, json as _json
_result = {"status": "ok", "answers": []}
def _save():
    import notebookutils
    notebookutils.fs.put("Files/demo1_full_smoke.json", _json.dumps(_result, indent=1), True)

PROMPTS = [
    "Which stores contributed most to the decline? Show store-level values for both years.",
    "How many stores show low inventory in the available snapshot? Can this prove that "
    "stock shortages caused the whole-month decline?",
]

try:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-q",
                    "fabric-data-agent-sdk"], check=True)

    from fabric.dataagent.client import FabricOpenAIResponses
    client = FabricOpenAIResponses(artifact_name="RetailEvidence")

    prev_id = None
    for q in PROMPTS:
        kwargs = {"input": q}
        if prev_id:
            kwargs["previous_response_id"] = prev_id
        resp = client.responses.create(**kwargs)
        prev_id = resp.id
        _result["answers"].append({"q": q, "a": resp.output_text})
        _save()
except Exception:
    _result["status"] = "error"
    _result["answers"].append({"q": "ERROR", "a": traceback.format_exc()})

_save()
print(_result["status"])
